# 실행코드

In [ ]:
# ============================================================
# SmartWarehouse Final Clean Code
# top300 + 5-model ensemble + raw target clipping + global calibration
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor


# ============================================================
# 0. 기본 설정
# ============================================================

TARGET = "avg_delay_minutes_next_30m"
DATA_DIR = "./data"

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("train shape:", train.shape)
print("test shape :", test.shape)


# ============================================================
# 1. Group-aware Feature Engineering v2
# ============================================================

def add_features_group_aware_v2(df):
    df = df.copy()
    group_key = "scenario_id"
    new_features = {}

    if "robot_active" in df.columns and "robot_idle" in df.columns:
        new_features["active_idle_ratio"] = df["robot_active"] / (df["robot_idle"] + 1)

    if "order_inflow_15m" in df.columns and "robot_active" in df.columns:
        new_features["orders_per_robot"] = df["order_inflow_15m"] / (df["robot_active"] + 1)

    if "charge_queue_length" in df.columns and "robot_charging" in df.columns:
        new_features["charge_pressure"] = df["charge_queue_length"] / (df["robot_charging"] + 1)

    if "congestion_score" in df.columns and "robot_active" in df.columns:
        new_features["congestion_robot"] = df["congestion_score"] * df["robot_active"]

    if "battery_std" in df.columns and "robot_charging" in df.columns:
        new_features["battery_charge_interaction"] = df["battery_std"] * df["robot_charging"]

    group_cols = [
        "order_inflow_15m", "robot_active", "congestion_score",
        "battery_std", "robot_charging", "charge_queue_length",
        "max_zone_density", "sku_concentration", "manual_override_ratio",
        "pack_utilization", "avg_trip_distance", "robot_utilization",
        "robot_idle", "battery_mean", "outbound_truck_wait_min"
    ]

    for col in group_cols:
        if col in df.columns:
            grp = df.groupby(group_key)[col]

            grp_mean = grp.transform("mean")
            grp_std = grp.transform("std").fillna(0)
            grp_min = grp.transform("min")
            grp_max = grp.transform("max")
            grp_median = grp.transform("median")

            new_features[f"{col}_grp_mean"] = grp_mean
            new_features[f"{col}_grp_std"] = grp_std
            new_features[f"{col}_grp_min"] = grp_min
            new_features[f"{col}_grp_max"] = grp_max
            new_features[f"{col}_grp_median"] = grp_median
            new_features[f"{col}_diff_mean"] = df[col] - grp_mean
            new_features[f"{col}_diff_median"] = df[col] - grp_median
            new_features[f"{col}_ratio_mean"] = df[col] / (grp_mean + 1e-6)
            new_features[f"{col}_zscore"] = (df[col] - grp_mean) / (grp_std + 1e-6)
            new_features[f"{col}_rank"] = grp.rank(method="average")
            new_features[f"{col}_pct_rank"] = grp.rank(pct=True)
            new_features[f"{col}_range_pos"] = (df[col] - grp_min) / (grp_max - grp_min + 1e-6)

    interaction_pairs = [
        ("order_inflow_15m", "max_zone_density"),
        ("order_inflow_15m", "robot_active"),
        ("order_inflow_15m", "congestion_score"),
        ("order_inflow_15m", "sku_concentration"),
        ("robot_active", "congestion_score"),
        ("robot_active", "manual_override_ratio"),
        ("robot_utilization", "congestion_score"),
        ("robot_utilization", "battery_std"),
        ("battery_std", "congestion_score"),
        ("charge_queue_length", "robot_charging"),
        ("pack_utilization", "sku_concentration"),
        ("avg_trip_distance", "order_inflow_15m")
    ]

    for a, b in interaction_pairs:
        if a in df.columns and b in df.columns:
            new_features[f"{a}_x_{b}"] = df[a] * df[b]
            new_features[f"{a}_div_{b}"] = df[a] / (df[b] + 1e-6)

    feature_df = pd.DataFrame(new_features, index=df.index)
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan)

    df = pd.concat([df, feature_df], axis=1)
    df = df.copy()

    return df


# ============================================================
# 2. Time Feature v2
# ============================================================

def add_time_features_v2(df):
    df = df.copy()

    if "scenario_id" not in df.columns:
        print("scenario_id 없음 → time feature 생략")
        return df

    df["_orig_order"] = np.arange(len(df))

    possible_time_cols = [
        "timestamp", "time", "datetime", "minute", "timestep",
        "step", "slot", "time_idx"
    ]

    time_col = None
    for col in possible_time_cols:
        if col in df.columns:
            time_col = col
            break

    if time_col is not None:
        df = df.sort_values(["scenario_id", time_col]).copy()
        print("time feature 기준 컬럼:", time_col)
    else:
        df = df.sort_values(["scenario_id", "_orig_order"]).copy()
        print("time feature 기준: scenario_id 내부 원본 행 순서")

    key_cols = [
        "order_inflow_15m",
        "robot_active",
        "robot_idle",
        "robot_charging",
        "congestion_score",
        "max_zone_density",
        "charge_queue_length",
        "battery_std",
        "battery_mean",
        "sku_concentration",
        "manual_override_ratio",
        "pack_utilization",
        "avg_trip_distance",
        "robot_utilization",
        "outbound_truck_wait_min"
    ]

    lags = [1, 2, 3, 5, 7]
    rolls = [3, 5, 7, 10]

    new_features = {}

    for col in key_cols:
        if col not in df.columns:
            continue

        grp = df.groupby("scenario_id")[col]

        for lag in lags:
            lag_val = grp.shift(lag)
            new_features[f"{col}_lag{lag}"] = lag_val
            new_features[f"{col}_diff_lag{lag}"] = df[col] - lag_val

        shifted = grp.shift(1)

        for window in rolls:
            new_features[f"{col}_roll{window}_mean"] = (
                shifted.groupby(df["scenario_id"])
                .rolling(window, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )

            new_features[f"{col}_roll{window}_std"] = (
                shifted.groupby(df["scenario_id"])
                .rolling(window, min_periods=1)
                .std()
                .reset_index(level=0, drop=True)
                .fillna(0)
            )

        prev = grp.shift(1)
        new_features[f"{col}_pct_change1"] = (df[col] - prev) / (prev.abs() + 1e-6)

    feature_df = pd.DataFrame(new_features, index=df.index)
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan)

    df = pd.concat([df, feature_df], axis=1)

    df = df.sort_values("_orig_order").drop(columns=["_orig_order"])
    df = df.copy()

    return df


# ============================================================
# 3. object 컬럼 인코딩
# ============================================================

def encode_object_columns(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    common_cols = train_df.columns.intersection(test_df.columns)

    for col in common_cols:
        if train_df[col].dtype == "object" or test_df[col].dtype == "object":
            combined = pd.concat([train_df[col], test_df[col]], axis=0)
            combined = combined.astype(str).fillna("missing")

            codes, _ = pd.factorize(combined)

            train_df[col] = codes[:len(train_df)]
            test_df[col] = codes[len(train_df):]

    return train_df, test_df


# ============================================================
# 4. Feature 적용
# ============================================================

train = add_features_group_aware_v2(train)
test = add_features_group_aware_v2(test)

train = add_time_features_v2(train)
test = add_time_features_v2(test)

train, test = encode_object_columns(train, test)

print("after feature train shape:", train.shape)
print("after feature test shape :", test.shape)


# ============================================================
# 5. Feature Selection
# layout_id 유지
# ============================================================

drop_cols = [TARGET, "ID", "scenario_id"]

X_all = train.drop(columns=drop_cols, errors="ignore")
y_log = np.log1p(train[TARGET])

test_feature_cols = test.drop(columns=["ID", "scenario_id"], errors="ignore").columns
common_features = X_all.columns.intersection(test_feature_cols)

X_all = X_all[common_features]

importance_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=64,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

importance_model.fit(X_all, y_log)

importances = pd.Series(
    importance_model.feature_importances_,
    index=X_all.columns
).sort_values(ascending=False)

top300 = importances.head(300).index.tolist()

best_feature_name = "top300"
best_feature_cols = top300

print("전체 후보 feature 수:", len(importances))
print("사용 feature set:", best_feature_name)
print("사용 feature 수:", len(best_feature_cols))
print(importances.head(50))


# ============================================================
# 6. 모델 파라미터
# ============================================================

CAT_LOG_PARAMS = {
    "iterations": 2000,
    "learning_rate": 0.02,
    "depth": 9,
    "l2_leaf_reg": 7,
    "random_strength": 1.0
}

CAT_RAW_BASE_PARAMS = {
    "loss_function": "MAE",
    "iterations": 2000,
    "learning_rate": 0.02,
    "depth": 9,
    "l2_leaf_reg": 7,
    "random_strength": 1.0
}

CAT_RAW_STRONG_PARAMS = {
    "loss_function": "MAE",
    "iterations": 2500,
    "learning_rate": 0.016,
    "depth": 9,
    "l2_leaf_reg": 9,
    "random_strength": 0.8
}

LGB_RAW_PARAMS = {
    "objective": "mae",
    "n_estimators": 2000,
    "learning_rate": 0.02,
    "num_leaves": 128,
    "max_depth": 10,
    "min_child_samples": 40,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
}

XGB_LOG_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 1800,
    "learning_rate": 0.018,
    "max_depth": 8,
    "min_child_weight": 20,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0
}


# ============================================================
# 7. Target clipping / Weight search / Calibration 함수
# ============================================================

RAW_TARGET_CLIP_Q = 99.5

SELECTED_MODELS = [
    "cat_log",
    "lgb_raw",
    "cat_raw_base",
    "cat_raw_strong",
    "xgb_log"
]


def get_clipped_target(y, q=99.5):
    if q is None:
        return y.copy(), None

    upper = np.percentile(y, q)
    y_clip = np.clip(y, 0, upper)

    return y_clip, upper


def evaluate_weights_batched(pred_arrays, y_true, weights_array, batch_size=64):
    P = np.vstack(pred_arrays)
    y = y_true.reshape(1, -1)

    best_score = 999
    best_weight = None
    rows = []

    for start in range(0, len(weights_array), batch_size):
        W = weights_array[start:start + batch_size]
        preds = W @ P
        maes = np.mean(np.abs(preds - y), axis=1)

        for i, score in enumerate(maes):
            w = W[i]

            rows.append({
                "weights": tuple(w),
                "mae": score
            })

            if score < best_score:
                best_score = score
                best_weight = w.copy()

    result_df = pd.DataFrame(rows).sort_values("mae")
    return best_weight, best_score, result_df


def make_weight_candidates(n_models=5, n_random=15000, seed=42):
    rng = np.random.default_rng(seed)

    weights = []

    weights.append(np.array([0.23541643, 0.11552829, 0.09731237, 0.30532822, 0.24641468]))
    weights.append(np.array([0.20, 0.10, 0.10, 0.35, 0.25]))
    weights.append(np.array([0.20, 0.10, 0.15, 0.30, 0.25]))
    weights.append(np.array([0.25, 0.10, 0.10, 0.30, 0.25]))
    weights.append(np.ones(n_models) / n_models)

    random_weights = rng.dirichlet(alpha=np.ones(n_models), size=n_random)
    weights.extend(list(random_weights))

    weights = np.array(weights)
    weights = weights / weights.sum(axis=1, keepdims=True)

    return weights


def refine_weights_around_best(best_weight, n_random=12000, radius=0.05, seed=777):
    rng = np.random.default_rng(seed)

    candidates = []

    for _ in range(n_random):
        noise = rng.normal(0, radius, size=len(best_weight))
        w = best_weight + noise
        w = np.clip(w, 0, None)

        if w.sum() == 0:
            continue

        w = w / w.sum()
        candidates.append(w)

    candidates.append(best_weight)

    return np.array(candidates)


def apply_scale_bias_clip(pred, scale, bias, clip_q):
    pred = pred * scale + bias
    pred = np.clip(pred, 0, None)

    upper = None
    if clip_q is not None:
        upper = np.percentile(pred, clip_q)
        pred = np.clip(pred, 0, upper)

    return pred, upper


def search_scale_bias_clip(oof_pred, y_true):
    scales = np.arange(0.88, 1.061, 0.01)
    biases = np.arange(-2.0, 2.01, 0.25)
    clip_qs = [97.5, 98.0, 98.5, 99.0, 99.5, None]

    rows = []
    best_score = 999
    best_params = None

    for scale in scales:
        for bias in biases:
            for clip_q in clip_qs:
                pred, upper = apply_scale_bias_clip(oof_pred, scale, bias, clip_q)
                mae = mean_absolute_error(y_true, pred)

                rows.append({
                    "scale": scale,
                    "bias": bias,
                    "clip_q": clip_q,
                    "clip_upper": upper,
                    "mae": mae
                })

                if mae < best_score:
                    best_score = mae
                    best_params = {
                        "scale": scale,
                        "bias": bias,
                        "clip_q": clip_q,
                        "clip_upper": upper
                    }

    result_df = pd.DataFrame(rows).sort_values("mae")

    print("\n===== Scale / Bias / Clip Calibration =====")
    print("best params:", best_params)
    print("best calibration MAE:", best_score)
    print(result_df.head(20))

    return best_params, best_score, result_df


# ============================================================
# 8. CV 실행
# ============================================================

def run_5model_cv_with_calibration(feature_cols):
    X = train[feature_cols]
    y_raw = train[TARGET]
    y_log = np.log1p(train[TARGET])
    groups = train["scenario_id"]

    gkf = GroupKFold(n_splits=5)

    oof = {name: np.zeros(len(train)) for name in SELECTED_MODELS}

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y_log, groups), 1):
        print(f"\n===== Fold {fold} =====")

        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]

        y_tr_log = y_log.iloc[tr_idx]
        y_tr_raw = y_raw.iloc[tr_idx]
        y_val_raw = y_raw.iloc[val_idx]

        y_tr_raw_clip, fold_raw_upper = get_clipped_target(
            y_tr_raw.values,
            RAW_TARGET_CLIP_Q
        )

        print(f"raw target clip upper fold {fold}: {fold_raw_upper}")

        cat_log = CatBoostRegressor(
            **CAT_LOG_PARAMS,
            random_seed=42,
            verbose=0,
            allow_writing_files=False
        )

        lgb_raw = LGBMRegressor(**LGB_RAW_PARAMS)

        cat_raw_base = CatBoostRegressor(
            **CAT_RAW_BASE_PARAMS,
            random_seed=42,
            verbose=0,
            allow_writing_files=False
        )

        cat_raw_strong = CatBoostRegressor(
            **CAT_RAW_STRONG_PARAMS,
            random_seed=42,
            verbose=0,
            allow_writing_files=False
        )

        xgb_log = XGBRegressor(**XGB_LOG_PARAMS)

        cat_log.fit(X_tr, y_tr_log)
        lgb_raw.fit(X_tr, y_tr_raw_clip)
        cat_raw_base.fit(X_tr, y_tr_raw_clip)
        cat_raw_strong.fit(X_tr, y_tr_raw_clip)
        xgb_log.fit(X_tr, y_tr_log)

        preds = {
            "cat_log": np.expm1(cat_log.predict(X_val)),
            "lgb_raw": lgb_raw.predict(X_val),
            "cat_raw_base": cat_raw_base.predict(X_val),
            "cat_raw_strong": cat_raw_strong.predict(X_val),
            "xgb_log": np.expm1(xgb_log.predict(X_val))
        }

        for model_name in SELECTED_MODELS:
            preds[model_name] = np.clip(preds[model_name], 0, None)
            oof[model_name][val_idx] = preds[model_name]

            print(
                f"{model_name} MAE:",
                mean_absolute_error(y_val_raw, preds[model_name])
            )

    y_true = y_raw.values

    print("\n===== 단일 모델 OOF =====")
    for model_name in SELECTED_MODELS:
        print(model_name, ":", mean_absolute_error(y_true, oof[model_name]))

    pred_arrays = [oof[name] for name in SELECTED_MODELS]

    candidates = make_weight_candidates(
        n_models=len(SELECTED_MODELS),
        n_random=15000,
        seed=42
    )

    best_w, best_score, weight_df = evaluate_weights_batched(
        pred_arrays=pred_arrays,
        y_true=y_true,
        weights_array=candidates,
        batch_size=64
    )

    refined_candidates = refine_weights_around_best(
        best_weight=best_w,
        n_random=12000,
        radius=0.05,
        seed=777
    )

    refined_w, refined_score, refined_df = evaluate_weights_batched(
        pred_arrays=pred_arrays,
        y_true=y_true,
        weights_array=refined_candidates,
        batch_size=64
    )

    if refined_score < best_score:
        best_w = refined_w
        best_score = refined_score
        best_df = refined_df
    else:
        best_df = weight_df

    print("\n===== Weight Search Result =====")
    print("BEST_WEIGHTS:", best_w)
    print("BEST_WEIGHT_MAE:", best_score)
    print(best_df.head(20))

    oof_blend = np.zeros(len(train), dtype=float)

    for w, name in zip(best_w, SELECTED_MODELS):
        oof_blend += w * oof[name]

    base_mae = mean_absolute_error(y_true, oof_blend)

    print("\n===== Weighted OOF =====")
    print("base ensemble MAE:", base_mae)

    calib_params, calib_mae, calib_df = search_scale_bias_clip(oof_blend, y_true)

    print("\n===== Final CV Result =====")
    print("base MAE :", base_mae)
    print("calib MAE:", calib_mae)

    return {
        "model_names": SELECTED_MODELS,
        "oof": oof,
        "best_weights": best_w,
        "base_mae": base_mae,
        "calib_params": calib_params,
        "calib_mae": calib_mae,
        "final_oof_mae": calib_mae,
        "weight_df": best_df,
        "calib_df": calib_df
    }


calib_result = run_5model_cv_with_calibration(best_feature_cols)

MODEL_NAMES = calib_result["model_names"]
BEST_WEIGHTS = calib_result["best_weights"]
BEST_BASE_MAE = calib_result["base_mae"]
BEST_CALIB_PARAMS = calib_result["calib_params"]
BEST_FINAL_OOF_MAE = calib_result["final_oof_mae"]

print("\n===== 최종 CV 요약 =====")
print("MODEL_NAMES:", MODEL_NAMES)
print("BEST_WEIGHTS:", BEST_WEIGHTS)
print("base MAE:", BEST_BASE_MAE)
print("calib params:", BEST_CALIB_PARAMS)
print("FINAL OOF MAE:", BEST_FINAL_OOF_MAE)


# ============================================================
# 9. 최종 모델 학습 + 제출 파일 생성
# ============================================================

X_train = train[best_feature_cols]
X_test = test[best_feature_cols]

y_train_log = np.log1p(train[TARGET])
y_train_raw = train[TARGET]

y_train_raw_clip, full_raw_upper = get_clipped_target(
    y_train_raw.values,
    RAW_TARGET_CLIP_Q
)

print("\nfull train raw target clip upper:", full_raw_upper)

cat_log_model = CatBoostRegressor(
    **CAT_LOG_PARAMS,
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)

lgb_raw_model = LGBMRegressor(**LGB_RAW_PARAMS)

cat_raw_base_model = CatBoostRegressor(
    **CAT_RAW_BASE_PARAMS,
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)

cat_raw_strong_model = CatBoostRegressor(
    **CAT_RAW_STRONG_PARAMS,
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)

xgb_log_model = XGBRegressor(**XGB_LOG_PARAMS)

cat_log_model.fit(X_train, y_train_log)
lgb_raw_model.fit(X_train, y_train_raw_clip)
cat_raw_base_model.fit(X_train, y_train_raw_clip)
cat_raw_strong_model.fit(X_train, y_train_raw_clip)
xgb_log_model.fit(X_train, y_train_log)

test_preds = [
    np.expm1(cat_log_model.predict(X_test)),
    lgb_raw_model.predict(X_test),
    cat_raw_base_model.predict(X_test),
    cat_raw_strong_model.predict(X_test),
    np.expm1(xgb_log_model.predict(X_test))
]

test_preds = [np.clip(p, 0, None) for p in test_preds]

final_pred_base = np.zeros(len(test), dtype=float)

for w, p in zip(BEST_WEIGHTS, test_preds):
    final_pred_base += w * p

final_pred_base = np.clip(final_pred_base, 0, None)

print("\n===== Test prediction before calibration =====")
print(pd.Series(final_pred_base).describe())

final_pred, global_upper = apply_scale_bias_clip(
    final_pred_base,
    BEST_CALIB_PARAMS["scale"],
    BEST_CALIB_PARAMS["bias"],
    BEST_CALIB_PARAMS["clip_q"]
)

print("\n===== Test prediction after calibration =====")
print("global clip upper:", global_upper)
print(pd.Series(final_pred).describe())

sample_submission[TARGET] = final_pred

output_path = "submission_top300_5model_rawclip_global_calib.csv"
sample_submission.to_csv(output_path, index=False)

print("저장 완료:", output_path)
print("FINAL OOF MAE:", BEST_FINAL_OOF_MAE)
print("calib params:", BEST_CALIB_PARAMS)
print(sample_submission.head())
print(sample_submission[TARGET].describe())